# BigQuery MCP - Semantic Search Verification

This notebook verifies the implementation of the `ekb_semantic_search` tool within the BigQuery MCP server. It tests the `BigQueryManager.semantic_search` method directly using Google Application Default Credentials (ADC).

In [ ]:
import sys
import os
import json

# Append repository root to sys.path to enable importing local modules
# The notebook is located in notebooks/big_query/, so we go up two levels.
repo_root = os.path.abspath("../..")
if repo_root not in sys.path:
    sys.path.append(repo_root)

import google.auth
from mcp_servers.big_query.app.bq_client import BigQueryManager
from mcp_servers.big_query.app.schemas import SemanticSearchRequest, AvailableProject

print(f"Repository root added to path: {repo_root}")

## 1. Initialize BigQuery Manager

We initialize the manager using ADC. In a production MCP environment, this would use a delegated OAuth access token from the user.

In [ ]:
try:
    # Attempt to get default credentials
    credentials, project_id = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    
    # Initialize the manager
    # We use the project defined in AvailableProject.DEV or the one from credentials
    target_project = AvailableProject.DEV
    bq_manager = BigQueryManager(creds=credentials, default_project=target_project)
    
    print(f"Initialized BigQueryManager for project: {target_project}")
except Exception as e:
    print(f"Failed to initialize BigQueryManager: {e}")
    print("Make sure you have run 'gcloud auth application-default login'.")

## 2. Execute Semantic Search

We perform a test search against the Enterprise Knowledge Base.

In [ ]:
# Define a test search request
search_request = SemanticSearchRequest(
    project_id=AvailableProject.DEV,
    query="Explain the security protocols and architecture of the Research-Agent project.",
    top_k=5,
    # domain="it", # Optional filter
)

print(f"Performing semantic search for: '{search_request.query}'...\n")

try:
    results = bq_manager.semantic_search(search_request)
    
    if not results:
        print("No results found. Ensure the 'knowledge_base' dataset and 'documents_chunks' table exist and are populated.")
    else:
        print(f"Found {len(results)} relevant chunks:\n")
        for i, res in enumerate(results):
            distance = res.get("distance", 0.0)
            filename = res.get("filename", "Unknown")
            page = res.get("page_number", 0)
            content = res.get("chunk_data", "")
            
            print(f"--- Result {i+1} [Distance: {distance:.4f}] ---")
            print(f"Source: {filename} (Page {page})")
            print(f"Snippet: {content[:300]}...")
            print(f"Metadata: Domain={res.get('domain')}, Trust={res.get('trust_level')}\n")
            
except Exception as e:
    print(f"Search failed: {e}")

## 3. Verify Filters

Test if optional metadata filters are correctly applied.

In [ ]:
# Test with a specific domain filter
filter_request = SemanticSearchRequest(
    project_id=AvailableProject.DEV,
    query="security",
    top_k=3,
    domain="it"
)

print(f"Performing filtered search (domain='it')...\n")

try:
    filtered_results = bq_manager.semantic_search(filter_request)
    for res in filtered_results:
        print(f"Result: {res.get('filename')} | Domain: {res.get('domain')}")
except Exception as e:
    print(f"Filtered search failed: {e}")